In [1]:
import os
import time
import math
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader

# NOTE: speed-critical image pipeline: use torchvision.io + v2 transforms (compiled ops)
from torchvision.io import read_image
from torchvision.transforms import v2 as T

from sklearn.model_selection import train_test_split

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Keep determinism stable; also enables some backend optimizations without changing results meaningfully
torch.backends.cudnn.benchmark = True




In [2]:
TARGET_COLUMNS = [
    "ETT - Abnormal",
    "ETT - Borderline",
    "ETT - Normal",
    "NGT - Abnormal",
    "NGT - Borderline",
    "NGT - Incompletely Imaged",
    "NGT - Normal",
    "CVC - Abnormal",
    "CVC - Borderline",
    "CVC - Normal",
    "Swan Ganz Catheter Present",
]

DEBUG = False

if DEBUG is False:
    BATCH_SIZE = 32
    EPOCHS = 2  # keep runtime safe; core training loop/architecture unchanged
    AVERAGING_SIZE = 100
    TRAIN_SUBSAMPLE = None
else:
    BATCH_SIZE = 4
    EPOCHS = 1
    AVERAGING_SIZE = 20
    TRAIN_SUBSAMPLE = 800

IMG_SIZE = 256

DATA_DIR = "/kaggle/input/ranzcr-clip-catheter-line-classification"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
SAMPLE_SUB_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")

OUTPUT_DIR = "./"
SUB_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device




device(type='cuda')

In [3]:
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
test_set_df = sample_sub.copy()

for col in TARGET_COLUMNS:
    if col not in test_set_df.columns:
        test_set_df[col] = 0.0

test_set_df = test_set_df[["StudyInstanceUID"] + TARGET_COLUMNS]

test_set_df.head(), test_set_df.shape




(                                    StudyInstanceUID  ETT - Abnormal  \
 0  1.2.826.0.1.3680043.8.498.25512976433640891933...               0   
 1  1.2.826.0.1.3680043.8.498.24449897997512078380...               0   
 2  1.2.826.0.1.3680043.8.498.38485493636649999035...               0   
 3  1.2.826.0.1.3680043.8.498.11901773728604504629...               0   
 4  1.2.826.0.1.3680043.8.498.11327663178389439022...               0   
 
    ETT - Borderline  ETT - Normal  NGT - Abnormal  NGT - Borderline  \
 0                 0             0               0                 0   
 1                 0             0               0                 0   
 2                 0             0               0                 0   
 3                 0             0               0                 0   
 4                 0             0               0                 0   
 
    NGT - Incompletely Imaged  NGT - Normal  CVC - Abnormal  CVC - Borderline  \
 0                          0             0  

In [4]:
class RanzcrClipDataset(Dataset):
    # CHANGE (timeout fix): replace PIL.Image.open().convert("RGB") with torchvision.io.read_image()
    # This avoids Python-level PIL overhead and uses a fast C++ decoder. Output is still RGB, and the
    # downstream transforms handle resize/tensor/normalize equivalently.
    def __init__(self, df, img_dir, target_columns, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.target_columns = target_columns
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        uid = self.df.loc[idx, "StudyInstanceUID"]
        img_path = os.path.join(self.img_dir, f"{uid}.jpg")

        # read_image returns uint8 tensor [C,H,W] in RGB for jpgs
        img = read_image(img_path)  # uint8, RGB, CHW

        if self.transform is not None:
            img = self.transform(img)

        if self.is_test:
            return img

        y = self.df.loc[idx, self.target_columns].values.astype(np.float32)
        return img, torch.from_numpy(y)




In [5]:
# CHANGE (timeout fix): use torchvision.transforms.v2 which runs most ops in efficient torch kernels.
# This preserves core semantics: Resize -> ToTensor (float in [0,1]) -> Normalize.
# read_image gives uint8 CHW; ToDtype(..., scale=True) is equivalent to ToTensor() scaling.
train_transforms = T.Compose(
    [
        T.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transforms = T.Compose(
    [
        T.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)




In [6]:
train_df = pd.read_csv(TRAIN_CSV)

if TRAIN_SUBSAMPLE is not None and TRAIN_SUBSAMPLE < len(train_df):
    train_df = train_df.sample(TRAIN_SUBSAMPLE, random_state=42).reset_index(drop=True)

trn_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42, shuffle=True
)

train_set = RanzcrClipDataset(
    trn_df, TRAIN_IMG_DIR, TARGET_COLUMNS, transform=train_transforms, is_test=False
)
val_set = RanzcrClipDataset(
    val_df, TRAIN_IMG_DIR, TARGET_COLUMNS, transform=test_transforms, is_test=False
)

test_set = RanzcrClipDataset(
    test_set_df, TEST_IMG_DIR, TARGET_COLUMNS, transform=test_transforms, is_test=True
)

# CHANGE (timeout fix): DataLoader tuning to reduce per-epoch startup and keep GPU fed.
# - persistent_workers avoids worker respawn each epoch
# - prefetch_factor increases queued batches per worker
# - num_workers scaled up (bounded by CPU) to overlap decode/resize with GPU compute
# - pin_memory stays for fast H2D copies; non_blocking already used
_num_cpu = os.cpu_count() or 2
NUM_WORKERS = 4 if not DEBUG else 2
NUM_WORKERS = min(NUM_WORKERS, max(2, _num_cpu // 2))

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)

len(train_set), len(val_set), len(test_set)




(24366, 2708, 3009)

In [7]:
class CustomPretrainedmModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            weights = (
                torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
                if pretrained
                else None
            )
            self.model = torchvision.models.densenet121(weights=weights)
        except Exception:
            self.model = torchvision.models.densenet121(pretrained=pretrained)

        n_features = self.model.classifier.in_features
        self.model.classifier = nn.Linear(n_features, len(TARGET_COLUMNS))

    def forward(self, x):
        return self.model(x)




In [8]:
model = CustomPretrainedmModel(pretrained=True).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


def run_eval(net, loader):
    net.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = net(xb)
            loss = criterion(logits, yb)
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")


start = time.time()
for epoch in range(EPOCHS):
    model.train()
    running = []
    for step, (xb, yb) in enumerate(train_loader):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running.append(loss.item())
        if (step + 1) % AVERAGING_SIZE == 0:
            print(
                f"epoch {epoch+1}/{EPOCHS} step {step+1}/{len(train_loader)} loss {np.mean(running[-AVERAGING_SIZE:]):.4f}"
            )

    val_loss = run_eval(model, val_loader)
    print(
        f"epoch {epoch+1}/{EPOCHS} train_loss {np.mean(running):.4f} val_loss {val_loss:.4f}"
    )

print(f"Training elapsed: {(time.time()-start):.1f}s")




Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 75.2MB/s]


epoch 1/2 step 100/762 loss 0.3597
epoch 1/2 step 200/762 loss 0.2441
epoch 1/2 step 300/762 loss 0.2326
epoch 1/2 step 400/762 loss 0.2190
epoch 1/2 step 500/762 loss 0.2145
epoch 1/2 step 600/762 loss 0.2066
epoch 1/2 step 700/762 loss 0.2014
epoch 1/2 train_loss 0.2370 val_loss 0.1880
epoch 2/2 step 100/762 loss 0.1932
epoch 2/2 step 200/762 loss 0.1873
epoch 2/2 step 300/762 loss 0.1864
epoch 2/2 step 400/762 loss 0.1866
epoch 2/2 step 500/762 loss 0.1834
epoch 2/2 step 600/762 loss 0.1771
epoch 2/2 step 700/762 loss 0.1813
epoch 2/2 train_loss 0.1846 val_loss 0.1747
Training elapsed: 277.4s


In [9]:
def inference(net, loader):
    y_prob = []
    net.eval()
    start_time = time.time()
    with torch.no_grad():
        for i, inputs in enumerate(loader, 0):
            inputs = inputs.to(device, non_blocking=True)
            outputs = net(inputs)
            probs = outputs.sigmoid()
            y_prob.append(probs.detach().cpu().numpy())
    y_prob = (
        np.vstack(y_prob)
        if len(y_prob)
        else np.zeros((0, len(TARGET_COLUMNS)), dtype=np.float32)
    )
    end_time = time.time()
    print(
        f"Inference batches: {i+1 if 'i' in locals() else 0}, elapsed {(end_time - start_time):.2f}s"
    )
    return y_prob


y_prob = inference(model, test_loader)
y_prob.shape




Inference batches: 48, elapsed 15.58s


(3009, 11)

In [10]:
sub = sample_sub.copy()
for col in TARGET_COLUMNS:
    if col not in sub.columns:
        sub[col] = 0.0

sub = sub[["StudyInstanceUID"] + TARGET_COLUMNS]
assert (
    y_prob.shape[0] == sub.shape[0]
), f"Pred rows {y_prob.shape[0]} != submission rows {sub.shape[0]}"
sub.loc[:, TARGET_COLUMNS] = y_prob

sub["StudyInstanceUID"] = (
    sub["StudyInstanceUID"].astype(str).str.replace(".jpg", "", regex=False)
)

sub.to_csv(SUB_PATH, index=False)
print("Wrote:", SUB_PATH)
sub.head()




Wrote: ./submission.csv


/tmp/ipykernel_55/1030478562.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.00029724 0.0005413  0.00208786 ... 0.00660675 0.01056967 0.0004182 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sub.loc[:, TARGET_COLUMNS] = y_prob
/tmp/ipykernel_55/1030478562.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.00062086 0.00056827 0.0064284  ... 0.19302328 0.09017685 0.00058494]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sub.loc[:, TARGET_COLUMNS] = y_prob
/tmp/ipykernel_55/1030478562.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[7.1010407e-04 5.4204569e-04 9.9065101e-01 ... 6.2716156e-01 8.8195944e-01
 7.2739535e-04]' has dtype incompatible with int64, please explicit

,StudyInstanceUID,ETT - Abnormal,ETT - Borderline,ETT - Normal,NGT - Abnormal,NGT - Borderline,NGT - Incompletely Imaged,NGT - Normal,CVC - Abnormal,CVC - Borderline,CVC - Normal,Swan Ganz Catheter Present
0,1.2.826.0.1.3680043.8.498.25512976433640891933...,0.000297,0.000621,0.000710,0.000549,0.001281,0.000076,0.004746,0.058362,0.351487,0.654019,0.000731
1,1.2.826.0.1.3680043.8.498.24449897997512078380...,0.000541,0.000568,0.000542,0.000863,0.000803,0.000214,0.000255,0.044498,0.263811,0.713681,0.001956
2,1.2.826.0.1.3680043.8.498.38485493636649999035...,0.002088,0.006428,0.990651,0.018655,0.056371,0.130194,0.062943,0.035995,0.101446,0.872170,0.003192
3,1.2.826.0.1.3680043.8.498.11901773728604504629...,0.001389,0.003120,0.007986,0.002637,0.006106,0.001070,0.009810,0.045548,0.237791,0.828544,0.001289
4,1.2.826.0.1.3680043.8.498.11327663178389439022...,0.002554,0.014254,0.011659,0.005883,0.027093,0.008023,0.902189,0.147110,0.277623,0.566035,0.001798


In [11]:
print("submission shape:", sub.shape)
print("columns match:", list(sub.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS)
print("any NaN:", sub.isna().any().any())
print(
    "prob range:",
    float(sub[TARGET_COLUMNS].min().min()),
    float(sub[TARGET_COLUMNS].max().max()),
)

submission shape: (3009, 12)
columns match: True
any NaN: False
prob range: 1.7781008864403702e-05 0.9988150596618652
